In [1]:
import math
import sympy as sp

In [2]:
eps = 1e-6
A, B = 0.0, 1.0

Подготавливаем выражение для производных

In [3]:
X = sp.Symbol("x")
FX = X ** 4 + sp.exp(-X)
DFX = sp.diff(FX, X)
DDFX = sp.diff(DFX, X)

Превращаем выражения в функции, которые считают числа

In [4]:
f = sp.lambdify(X, FX, "math")
df = sp.lambdify(X, DFX, "math")
d2f = sp.lambdify(X, DDFX, "math")

print("f(x) =", FX)
print("f'(x) =", DFX)
print("f''(x) =", DDFX)
print("f(0) = %.6f, f(1) = %.6f" % (f(A), f(B)))

f(x) = x**4 + exp(-x)
f'(x) = 4*x**3 - exp(-x)
f''(x) = 12*x**2 + exp(-x)
f(0) = 1.000000, f(1) = 1.367879


Проверка применимости методов

In [5]:
n = 1000
xs = [A + (B - A) * i / n for i in range(n + 1)]

print("f'(0) = %.6f  (должна быть < 0)" % df(A))
print("f'(1) = %.6f  (должна быть > 0)" % df(B))

d2_min = min(d2f(t) for t in xs)
if d2_min > 0:
    print("min f''(x) на [0;1] = %.6f > 0, функция выпуклая, минимум один" % d2_min)
else:
    print("f''(x) не везде > 0, методы могут не подойти")

L = max(abs(df(t)) for t in xs)
print("L = max|f'| = %.6f" % L)

f'(0) = -1.000000  (должна быть < 0)
f'(1) = 3.632121  (должна быть > 0)
min f''(x) на [0;1] = 0.979989 > 0, функция выпуклая, минимум один
L = max|f'| = 3.632121


Метод деления пополам

In [6]:
a = A
b = B
d = eps / 10
n = 0
while b - a > eps:
    x1 = (a + b - d) / 2
    x2 = (a + b + d) / 2
    if f(x1) < f(x2):
        b = x2
    else:
        a = x1
    n += 1
x = (a + b) / 2

print("Деление пополам: x = %.6f  f = %.6f  итераций = %d" % (x, f(x), n))

Деление пополам: x = 0.528252  f = 0.667504  итераций = 21


Метод касательных

In [7]:
a = A
b = B
n = 0
while b - a > eps:
    xm = (f(b) - f(a) + df(a) * a - df(b) * b) / (df(a) - df(b))
    g = df(xm)
    if abs(g) < eps:
        a = xm
        b = xm
    elif g > 0:
        b = xm
    else:
        a = xm
    n += 1
x = (a + b) / 2

print("Метод касательных: x = %.6f  f = %.6f  итераций = %d" % (x, f(x), n))

Метод касательных: x = 0.528252  f = 0.667504  итераций = 20


Метод ломаных

In [8]:
L = max(abs(df(A + (B - A) * i / 1000)) for i in range(1001))
xs = [A, B]
ys = [f(A), f(B)]
n = 0
while True:
    m = 0
    rmin = float("inf")
    for i in range(len(xs) - 1):
        r = (ys[i] + ys[i + 1]) / 2 - L * (xs[i + 1] - xs[i]) / 2
        if r < rmin:
            rmin = r
            m = i
    if xs[m + 1] - xs[m] <= eps:
        break
    xn = (xs[m] + xs[m + 1]) / 2 + (ys[m] - ys[m + 1]) / (2 * L)
    xs.insert(m + 1, xn)
    ys.insert(m + 1, f(xn))
    n += 1
k = ys.index(min(ys))
x = xs[k]

print("Метод ломаных: x = %.6f  f = %.6f  итераций = %d" % (x, f(x), n))

Метод ломаных: x = 0.528252  f = 0.667504  итераций = 5059


Метод Ньютона

In [9]:
x = B
n = 0
while True:
    xk = x - df(x) / d2f(x)
    n += 1
    if abs(xk - x) < eps:
        x = xk
        break
    x = xk

print("Метод Ньютона: x = %.6f  f = %.6f  итераций = %d" % (x, f(x), n))

Метод Ньютона: x = 0.528252  f = 0.667504  итераций = 6
